# 範例一：爬取維基百科各國人口表格

使用 `requests` 與 `BeautifulSoup` 讀取維基百科的國家人口統計表，再以 pandas DataFrame 顯示前 10 筆資料。

## 1. 安裝及載入模組

- `requests`：向網站發送 HTTP 請求，取得 HTML。
- `BeautifulSoup`：解析 HTML 標籤。
- `pandas`：整理及顯示表格資料。

In [ ]:
!pip -q install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd

## 2. 取得網頁 HTML

使用 `requests.get()` 取得網頁內容；`raise_for_status()` 可在 HTTP 請求失敗時顯示錯誤。

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; ColabWebScraper/1.0)"
}

response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()

print("HTTP 狀態碼：", response.status_code)
print(response.text[:500])  # 預覽 HTML 原始碼前 500 個字元

## 3. 使用 BeautifulSoup 解析 HTML

`find_all()` 可找出所有符合條件的標籤。這裡先找出 class 為 `wikitable` 的表格，並檢查找到幾個表格。

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

tables = soup.find_all("table", class_="wikitable")
print("找到的表格數量：", len(tables))

if not tables:
    raise ValueError("找不到 wikitable，網站結構可能已變更。")

## 4. 讀取第一個表格並轉成 DataFrame

以第一列的欄位標題作為欄名，逐列擷取儲存格文字。若網站表格結構改變，需重新檢查 HTML。

In [ ]:
table = tables[0]
rows = table.find_all("tr")

# 讀取標題列
column_names = [
    cell.get_text(" ", strip=True)
    for cell in rows[0].find_all(["th", "td"])
]

# 讀取資料列
data = []
for row in rows[1:]:
    cells = row.find_all(["th", "td"])
    values = [cell.get_text(" ", strip=True) for cell in cells]
    if values:
        data.append(values)

# 不同資料列若有欄位數差異，補齊後再建立 DataFrame
max_cols = max([len(column_names)] + [len(r) for r in data])
column_names = (column_names + [f"欄位{i}" for i in range(len(column_names)+1, max_cols+1)])[:max_cols]
data = [r + [""] * (max_cols - len(r)) for r in data]

df = pd.DataFrame(data, columns=column_names)
display(df.head(10))

## 5. 程式重點整理

| 指令 | 功能 |
|---|---|
| `requests.get(url)` | 向網站發送 GET 請求 |
| `response.text` | 取得 HTML 原始碼 |
| `BeautifulSoup(..., "html.parser")` | 建立 HTML 解析物件 |
| `find_all("tr")` | 找出所有資料列 |
| `find_all(["th", "td"])` | 找出標題或資料儲存格 |
| `get_text(" ", strip=True)` | 擷取標籤內文字並清除多餘空白 |
| `pd.DataFrame()` | 將資料整理成表格 |

**注意：** 網站可能調整版面或限制自動存取；若執行失敗，先檢查 HTTP 狀態碼及目前的 HTML 結構。